# Unit 5 · Linear regression

The 470-rental calendar forecast in Unit 4 gave the same prediction for every 8 a.m. workday. Can yesterday’s 8 a.m. count help? We will try a line, fit it, and compare it with that same baseline on later workdays.

Use [the teaching page](https://sharanharsoor.github.io/ml-learning-lab/ml-foundations/#unit-5) for the draggable line. This notebook reproduces the data, calculations, and one plot. Run the cells in order. Set the `BIKESHARE_ARCHIVE` environment variable or edit `local_archive` to use a local copy of the original ZIP.

The source is the Capital Bikeshare hourly dataset in [UCI Bike Sharing](https://doi.org/10.24432/C5W894) (Fanaee-T, 2013), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## 1. Pair a workday with the previous calendar day

A point has two values, yesterday’s 08:00 rental count `x` and today’s count `y`. Today is a workday. Yesterday can be any day, including Sunday. If the exact previous calendar date has no 08:00 row, set the target row aside for the line. The calendar baseline can still learn from its observed count.

On November 1, 2012, the usable input is **621 rentals** from October 31. A historical target can become a feature on the following day because it is already known then.

In [ ]:
import csv
import os
from datetime import date, timedelta
from io import BytesIO, TextIOWrapper
from pathlib import Path
from statistics import mean
from urllib.request import urlopen
from zipfile import ZipFile

uci_url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
local_archive = os.environ.get("BIKESHARE_ARCHIVE", "")

if local_archive:
    archive_bytes = Path(local_archive).read_bytes()
else:
    with urlopen(uci_url, timeout=30) as response:
        archive_bytes = response.read()

with ZipFile(BytesIO(archive_bytes)) as archive:
    with archive.open("hour.csv") as stream:
        hours = list(csv.DictReader(TextIOWrapper(stream, encoding="utf-8")))

eight_am = {
    date.fromisoformat(row["dteday"]): row for row in hours if row["hr"] == "8"
}
workdays = {day: row for day, row in eight_am.items() if row["workingday"] == "1"}

pairs = {}
missing_previous = []
for today, row in sorted(workdays.items()):
    yesterday = eight_am.get(today - timedelta(days=1))
    if yesterday is None:
        missing_previous.append(today)
        continue
    pairs[today] = (int(yesterday["cnt"]), int(row["cnt"]))

print(len(hours), "source hours")
print(len(pairs), "paired workdays")
print("Workdays with no exact prior 08:00 row:", ", ".join(map(str, missing_previous)))

## 2. Give each period one job

Split by the **target date**. Train through September 2012, validate in October, then keep November and December for the final test. Within each evaluation period, both rules predict on the same eligible workdays.

In [ ]:
october = date(2012, 10, 1)
november = date(2012, 11, 1)

train = [pair for day, pair in pairs.items() if day < october]
validation = [pair for day, pair in pairs.items() if october <= day < november]
test = [pair for day, pair in pairs.items() if day >= november]
x_focus, y_focus = pairs[november]

print(f"Training: {len(train)} pairs  |  Validation: {len(validation)} pairs  |  Test: {len(test)} pairs")
print(f"November 1 input: {x_focus} rentals from October 31, source row {eight_am[november - timedelta(days=1)]['instant']}")

## 3. Try a line and calculate the loss

The prediction is **ŷ = b + m × x**. With slope `m = 0.5`, 100 more rentals yesterday add 50 to the prediction. The intercept `b` sets the value at `x = 0` and moves the line up or down when the slope is held fixed.

For a training row, the residual is **y − ŷ**. A positive residual means the line predicts too low. Squaring keeps misses on opposite sides of the line from canceling. The **mean squared error (MSE)** is the sum of squared residuals divided by the number of rows.

Try changing either parameter. Compare the MSE of the three example rows with the MSE over all training rows. The full training MSE is the one the fit minimizes.

In [ ]:
def predict(x, slope, intercept):
    return intercept + slope * x

def mse(rows, slope, intercept):
    return mean((y - predict(x, slope, intercept)) ** 2 for x, y in rows)

trial_slope = 0.5
trial_intercept = 350
sample_dates = [date(2012, 9, day) for day in (17, 19, 28)]
sample = [pairs[day] for day in sample_dates]

print(f"{'Target date':<12} {'x':>5} {'y':>5} {'Prediction':>12} {'Residual':>10} {'Squared':>12}")
for day, (x, y) in zip(sample_dates, sample):
    estimate = predict(x, trial_slope, trial_intercept)
    residual = y - estimate
    print(f"{day.isoformat():<12} {x:5} {y:5} {estimate:12.2f} {residual:+10.2f} {residual**2:12,.2f}")

print(f"Three-row MSE: {mse(sample, trial_slope, trial_intercept):,.2f} rentals²")
print(f"Full training MSE: {mse(train, trial_slope, trial_intercept):,.2f} rentals²")

## 4. Fit the line

Ordinary least squares gives the slope **∑[(x − x̄)(y − ȳ)] ÷ ∑(x − x̄)²**. The numerator adds the products of departures from the two means. The denominator measures how much the input varies. With this slope, **b = ȳ − m × x̄** places the line through the point of the two means.

For a fixed set of rows, minimizing the sum of squared errors also minimizes the mean squared error. The denominator of MSE is the same for every line we compare.

In [ ]:
def fit_line(rows):
    x_mean = mean(x for x, y in rows)
    y_mean = mean(y for x, y in rows)
    cross_products = sum((x - x_mean) * (y - y_mean) for x, y in rows)
    input_squares = sum((x - x_mean) ** 2 for x, y in rows)
    slope = cross_products / input_squares
    intercept = y_mean - slope * x_mean
    return slope, intercept

slope, intercept = fit_line(train)
print(f"Training line: ŷ = {intercept:.4f} + {slope:.6f} × x")
print(f"Training MSE after fitting: {mse(train, slope, intercept):,.2f} rentals²")

## See the same fitted line

Each dot is one training pair. The orange trial line is the one from the worked table. The green line is fitted on all training pairs. The prediction is the height of the line at a given `x`. The residual is measured vertically.

In [ ]:
import matplotlib.pyplot as plt

x_values, y_values = zip(*train)
edges = [min(x_values), max(x_values)]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x_values, y_values, s=16, alpha=0.24, color="#52615F", label="Training pairs")
ax.plot(edges, [predict(x, trial_slope, trial_intercept) for x in edges],
        color="#B45A33", linewidth=2, linestyle="--", label="Trial line")
ax.plot(edges, [predict(x, slope, intercept) for x in edges],
        color="#076B61", linewidth=2, label="Fitted line")
ax.set(xlabel="Yesterday at 08:00 (rentals)", ylabel="Workday at 08:00 (rentals)",
       title="Training pairs through September 2012")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Choose a rule using October

The calendar baseline averages **all** earlier recorded 08:00 workday counts and rounds to a whole rental, as in Unit 4. It needs only the calendar. The line uses paired training rows and keeps its predictions at full precision. The two rules are scored on the same October pairs.

Training used MSE to fit the line. For this comparison we use **mean absolute error (MAE)**, the sum of absolute misses divided by the number of scored hours. Its units are rentals.

In [ ]:
def fit_calendar(cutoff):
    counts = [int(row["cnt"]) for day, row in workdays.items() if day < cutoff]
    return sum(counts), len(counts), round(mean(counts))

def mae(rows, slope, intercept):
    return mean(abs(y - predict(x, slope, intercept)) for x, y in rows)

total, count, calendar = fit_calendar(october)
validation_scores = {
    "Calendar baseline": mae(validation, 0, calendar),
    "Linear regression": mae(validation, slope, intercept),
}
winner = min(validation_scores, key=validation_scores.get)

print(f"Calendar baseline: {total:,} ÷ {count} rounds to {calendar} rentals")
for name, score in validation_scores.items():
    print(f"{name}: {score:.2f} rentals MAE")
print("Chosen on validation:", winner)

## 6. Refit through October, then compare on November and December

Use both training and validation history to refit after locking the modeling decision. Score both rules on the same eligible workdays in the final period. The input for any test date comes only from its exact prior day, even when that prior day falls inside the test period. This represents predicting one day at a time as new history arrives.

In [ ]:
final_slope, final_intercept = fit_line(train + validation)
total, count, final_calendar = fit_calendar(november)
calendar_test = mae(test, 0, final_calendar)
linear_test = mae(test, final_slope, final_intercept)
line_forecast = predict(x_focus, final_slope, final_intercept)

print(f"Refitted calendar baseline: {total:,} ÷ {count} rounds to {final_calendar} rentals")
print(f"Refitted line: ŷ = {final_intercept:.4f} + {final_slope:.6f} × x")
print(f"Calendar test MAE: {calendar_test * len(test):,.2f} ÷ {len(test)} = {calendar_test:.2f} rentals")
print(f"Line test MAE: {linear_test * len(test):,.2f} ÷ {len(test)} = {linear_test:.2f} rentals")
print(f"November 1: x = {x_focus}, recorded y = {y_focus}")
print(f"Calendar forecast: {final_calendar}, signed residual: {y_focus - final_calendar:+.0f} rentals")
print(f"Line forecast: {line_forecast:.2f}, signed residual: {y_focus - line_forecast:+.2f} rentals")

## Try it yourself

For November 1, suppose the slope is `0.5` and the intercept is `200`. Work out the prediction, residual, and squared error by hand before revealing the check. Which quantity has squared units?

<details>
<summary>Check your calculation</summary>

- Prediction = 200 + 0.5 × 621 = **510.50 rentals**.
- Residual = 680 − 510.50 = **+169.50 rentals**.
- Squared error = 169.50² = **28,730.25 rentals²**.

This trial line is for practice. The full fit used 453 earlier pairs before November 1.

</details>

The lower final MAE supports the line for these 40 workday hours and this one-day-ahead setup. It measures the observed forecast errors, and the slope describes the fitted association in these historical records.